# MS COCO CLIP ViT-B/32 → MobileNetV3-Large — Direct Logit-KD: Mobile Entry

Khóa checkpoint triển khai bằng Validation; không đọc Test và không thay đổi kết luận khoa học.

## Input bắt buộc — đúng 4 input

1. output `phase4_teacher_a_s1_logit_kd_seed123` (seed 123);
2. output `phase4_teacher_a_s1_logit_kd_three_seed_aggregate`;
3. output `01_MSCOCO_Data_Contract`;
4. output `02A_Phase2_Shared_Pipeline`;

Seed được khóa: **123**. CIDEr-D ×100: **98.73 → 100.65**.

Runtime: `CPU / Accelerator None / Internet Off`.

In [1]:
import csv, hashlib, json, os, shutil
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))

def now(): return datetime.now(timezone.utc).isoformat()
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (name, [str(x) for x in matches])
    return matches[0]
def nested(payload, dotted):
    value = payload
    for key in dotted.split("|"):
        value = value[key]
    return value

CFG = json.loads('{"short": "MSCOCO_CLIPS1", "title": "MS COCO CLIP ViT-B/32 → MobileNetV3-Large — Direct Logit-KD", "dataset": "MS COCO 2014", "method": "Direct Logit-KD", "teacher": "CLIP ViT-B/32", "student": "MobileNetV3-Large", "seed": 123, "result_name": "teacher_a_s1_logit_kd_seed123_vs_baseline.json", "aggregate_name": "teacher_a_s1_logit_kd_three_seed_summary.json", "checkpoint_name": "teacher_a_s1_logit_kd_seed123_best.pt", "checkpoint_hash_source": null, "baseline": 0.9872507558000245, "kd": 1.0065169992410692, "expected_keys": ["student_encoder_state", "student_decoder_state"], "vocab_name": "word_vocabulary_minfreq5.json", "vocab_array_key": "id_to_token", "vocab_size": 10080, "inputs": ["output `phase4_teacher_a_s1_logit_kd_seed123` (seed 123)", "output `phase4_teacher_a_s1_logit_kd_three_seed_aggregate`", "output `01_MSCOCO_Data_Contract`", "output `02A_Phase2_Shared_Pipeline`"]}')
EXPERIMENT_ID = "MOB_" + CFG["short"] + "_Entry"
OUT = Path(os.environ.get("KLCN_OUTPUT_ROOT", f"/kaggle/working/{EXPERIMENT_ID}"))
PACKAGE = OUT / "selected_checkpoint"
PACKAGE.mkdir(parents=True, exist_ok=True)

result_path = unique_named(CFG["result_name"])
aggregate_path = unique_named(CFG["aggregate_name"])
vocab_path = unique_named(CFG["vocab_name"])
result = read_json(result_path)
aggregate = read_json(aggregate_path)
vocab = read_json(vocab_path)

assert result.get("status", "complete") == "complete"
assert int(result.get("seed", CFG["seed"])) == CFG["seed"]
source_test_status = result.get("test_set_status", "sealed")
allowed_test_statuses = {"sealed", "previously_opened_for_locked_T3S3_only"}
assert source_test_status in allowed_test_statuses, source_test_status
assert result.get("test_set_used", False) is False
assert aggregate.get("status", "complete") == "complete"
assert len(vocab[CFG["vocab_array_key"]]) == CFG["vocab_size"]

checkpoint_matches = sorted(
    path for path in INPUT_ROOT.rglob(CFG["checkpoint_name"])
    if path.is_file()
)
assert len(checkpoint_matches) == 1, checkpoint_matches
checkpoint_path = checkpoint_matches[0]
if CFG["checkpoint_hash_source"]:
    expected_sha = nested(result, CFG["checkpoint_hash_source"])
    assert sha256(checkpoint_path) == expected_sha
else:
    expected_sha = sha256(checkpoint_path)

target_checkpoint = PACKAGE / "selected_best.pt"
target_vocab = PACKAGE / CFG["vocab_name"]
shutil.copy2(checkpoint_path, target_checkpoint)
shutil.copy2(vocab_path, target_vocab)

lock = {
    "schema_version": "mobile_expansion_entry_v1",
    "created_at_utc": now(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "model_id": CFG["short"],
    "teacher": CFG["teacher"],
    "student": CFG["student"],
    "dataset": CFG["dataset"],
    "method": CFG["method"],
    "selected_seed": CFG["seed"],
    "selection_reason": "stable seed near the three-seed central tendency; validation only",
    "baseline_cider_d": CFG["baseline"],
    "kd_cider_d": CFG["kd"],
    "metrics_x100": {
        "baseline_CIDEr_D": round(CFG["baseline"] * 100, 2),
        "KD_CIDEr_D": round(CFG["kd"] * 100, 2),
    },
    "checkpoint": {
        "relative_path": target_checkpoint.relative_to(OUT).as_posix(),
        "bytes": target_checkpoint.stat().st_size,
        "sha256": sha256(target_checkpoint),
    },
    "vocabulary": {
        "relative_path": target_vocab.relative_to(OUT).as_posix(),
        "size": CFG["vocab_size"],
        "sha256": sha256(target_vocab),
    },
    "aggregate_source": {"name": aggregate_path.name, "sha256": sha256(aggregate_path)},
    "selection_split": "validation",
    "source_test_set_status": source_test_status,
    "test_set_status": "not_used_by_this_experiment",
    "test_set_used": False,
    "next_phase": f"MOB_{CFG['short']}_ONNX_Preflight",
}
lock_path = OUT / f"MOB_{CFG['short']}_ENTRY_LOCK.json"
write_json(lock_path, lock)

with (OUT / "MANIFEST.csv").open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=["role", "relative_path", "bytes", "sha256"])
    writer.writeheader()
    for role, path in (("checkpoint", target_checkpoint), ("vocabulary", target_vocab), ("lock", lock_path)):
        writer.writerow({
            "role": role, "relative_path": path.relative_to(OUT).as_posix(),
            "bytes": path.stat().st_size, "sha256": sha256(path),
        })

print("Mobile entry lock       : PASSED")
print("Model                   :", CFG["teacher"], "->", CFG["student"])
print("Dataset / KD            :", CFG["dataset"], "/", CFG["method"])
print("Selected seed           :", CFG["seed"])
print("CIDEr-D baseline -> KD   :", round(CFG["baseline"]*100,2), "->", round(CFG["kd"]*100,2))
print("Test                    : SEALED / NOT READ")

Mobile entry lock       : PASSED
Model                   : CLIP ViT-B/32 -> MobileNetV3-Large
Dataset / KD            : MS COCO 2014 / Direct Logit-KD
Selected seed           : 123
CIDEr-D baseline -> KD   : 98.73 -> 100.65
Test                    : SEALED / NOT READ
